# Phase 2: DistilBERT on the clean splits

Runs on a Colab **GPU** runtime (Runtime → Change runtime type → T4 GPU), in about an hour.
It follows the plan fixed before training, `docs/experiments/phase2_analysis_plan.md`, and
ends by downloading `phase2_results.zip`.

1. **Data.** Downloads the Booking 515K CSV (size and SHA-256 checked) and builds the random
   and out-of-time splits. It stops unless they hold the same reviews, in the same order, as
   those Naive Bayes was scored on (`docs/experiments/results/split_views.json`).
2. **Training.** DistilBERT, full fine-tune and scratch LoRA, with the settings of the legacy
   run. The out-of-time split goes first, as it holds the primary comparison.
3. **Cost.** Latency on the CPU, the whole test set on the GPU, and size
   (`scripts/benchmark_latency.py`).
4. **Comparison.** `scripts/compare_split_models.py` applies the plan's rule.

**If Colab disconnects:** with `USE_DRIVE = True`, each finished run is kept in Google Drive.
Run the notebook again from the top; finished runs are skipped.

**Send back** `phase2_results.zip`. It holds metrics, logits, logs and manifests, not the
model weights.

In [1]:
# Keep each finished run in Google Drive (MyDrive/hotel-review-nlp-phase2), so a disconnect
# loses only the run in progress. Colab asks for permission once.
USE_DRIVE = True
# () is the plan. (43,) also trains the out-of-time full fine-tune with seed 43 (about 15
# more minutes); it is compared with Naive Bayes and reported beside the seed-42 result.
EXTRA_SEEDS = ()

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.")
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/hotel-review-nlp")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "--branch", "claude/funny-rubin-31xgtb",
                    "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
# The transformers version of the legacy run and of the CI.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.56.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("commit", GIT_COMMIT)

## 1. Data: the splits Naive Bayes was scored on

In [ ]:
import json

from reviewnlp.utils.experiments import fingerprint_splits
from reviewnlp.utils.seed import load_config


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


# The out-of-time split first: it holds the primary comparison.
SPLITS = {"time": ("configs/baselines_time.yaml", "configs/distilbert_v2_time.yaml"),
          "random": ("configs/baselines.yaml", "configs/distilbert_v2.yaml")}
views = json.loads(Path("docs/experiments/results/split_views.json").read_text(encoding="utf-8"))

run("python scripts/fetch_booking_515k.py")
for split, (data_config, model_config) in SPLITS.items():
    processed = load_config(model_config)["data"]["processed_dir"]
    assert processed == load_config(data_config)["data"]["processed_dir"]
    if not Path(processed, "test.parquet").exists():
        run(f"python -m reviewnlp.data.preprocess --config {data_config}")
    if fingerprint_splits(processed) != views[split]["splits"]:
        raise RuntimeError(f"The {split} split differs from the one Naive Bayes was scored on. Stopping.")
    print(f"{split}: the same reviews, in the same order, as in split_views.json")

## 2. Training (about 50 minutes on a T4)

For each split, the full fine-tune (about 15 minutes) and then the scratch LoRA (about 10).
A run whose `metrics.json` already exists is skipped.

In [ ]:
import shutil

import yaml

LORA = ["--r", "8", "--alpha", "16", "--dropout", "0.0", "--lr", "1e-4"]
RUNS = {}  # (split, name) -> (module, config, output directory, extra arguments)
for split, (_, model_config) in SPLITS.items():
    model_dir = load_config(model_config)["output"]["model_dir"]
    RUNS[split, "distilbert"] = ("reviewnlp.llm.train_distilbert", model_config, model_dir, [])
    RUNS[split, "distilbert_lora_scratch"] = (
        "reviewnlp.llm.train_distilbert_lora", model_config, f"{model_dir}_lora_scratch", LORA)
for seed in EXTRA_SEEDS:
    config = load_config("configs/distilbert_v2_time.yaml")
    config["seed"] = seed
    config["output"]["model_dir"] = f"runs/distilbert_v2_time_seed{seed}"
    path = Path(f"runs/configs/distilbert_v2_time_seed{seed}.yaml")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    RUNS["time", f"distilbert_seed{seed}"] = (
        "reviewnlp.llm.train_distilbert", str(path), config["output"]["model_dir"], [])

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    kept = Path("/content/drive/MyDrive/hotel-review-nlp-phase2")
    for _, _, out_dir, _ in RUNS.values():
        (kept / Path(out_dir).name).mkdir(parents=True, exist_ok=True)
        Path(out_dir).parent.mkdir(parents=True, exist_ok=True)
        if not Path(out_dir).exists() and not Path(out_dir).is_symlink():
            Path(out_dir).symlink_to(kept / Path(out_dir).name)


def train(split: str, name: str, module: str, config: str, out_dir: str, extra: list[str]) -> None:
    out = Path(out_dir)
    if (out / "metrics.json").exists():
        print(f"{split}/{name}: already trained, kept")
    else:
        out.mkdir(parents=True, exist_ok=True)
        request = {"commit": GIT_COMMIT, "gpu": torch.cuda.get_device_name(0), "module": module,
                   "config": config, "extra_args": extra}
        (out / "request.json").write_text(json.dumps(request, indent=1) + "\n", encoding="utf-8")
        command = [sys.executable, "-u", "-m", module, "--config", config, *extra]
        print("$", " ".join(command), flush=True)
        with open(out / "train.log", "w", encoding="utf-8") as log, subprocess.Popen(
                command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
        if process.returncode:
            raise RuntimeError(f"{split}/{name} failed; see {out / 'train.log'}")
    metrics = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
    if metrics["data"] != views[split]["splits"]:
        raise RuntimeError(f"{split}/{name} was trained or scored on other data")
    print(f"{split}/{name}: best dev macro-F1 {metrics['best_dev_macro_f1']:.4f}, "
          f"{metrics['training_seconds'] / 60:.1f} minutes")


for (split, name), (module, config, out_dir, extra) in RUNS.items():
    train(split, name, module, config, out_dir, extra)

## 3. Cost: latency, the whole test set, size (about 10 minutes)

Naive Bayes is trained again on the out-of-time split, only to be timed. The script times a
model only if it gives the predictions that were scored. If this step fails, the next one still
collects the results, without `latency.json`.

In [ ]:
PHASE2 = Path("runs/phase2/distilbert_v2")
try:
    run("python -m reviewnlp.baselines.classical --config configs/baselines_time.yaml")
    run(f"python scripts/benchmark_latency.py --split time --out {PHASE2}/latency.json")
except subprocess.CalledProcessError:
    print("The cost step failed (see above). Go on: the results are collected without latency.json.")

## 4. Collect, compare and download

In [ ]:
import hashlib
import platform
import zipfile
from importlib.metadata import version

from google.colab import files

KEEP = ["metrics.json", "test_logits.npy", "test_labels.npy", "dev_logits.npy", "dev_labels.npy",
        "request.json", "lora_scratch_config.json"]
for (split, name), (_, config, out_dir, _) in RUNS.items():
    target = PHASE2 / split / name
    target.mkdir(parents=True, exist_ok=True)
    for file in KEEP:
        if Path(out_dir, file).exists():
            shutil.copy2(Path(out_dir, file), target / file)
    if Path(out_dir, "train.log").exists():
        shutil.copy2(Path(out_dir, "train.log"), target / "train_log.txt")
    shutil.copy2(config, target / "run_config.yaml")

environment = {
    "commit": GIT_COMMIT, "gpu": torch.cuda.get_device_name(0), "python": platform.python_version(),
    "packages": {name: version(name) for name in
                 ("torch", "transformers", "tokenizers", "scikit-learn", "numpy", "pandas")},
    "extra_seeds": list(EXTRA_SEEDS)}
(PHASE2 / "environment.json").write_text(json.dumps(environment, indent=1) + "\n", encoding="utf-8")
try:
    run(f"python scripts/compare_split_models.py --artifacts {PHASE2} --out {PHASE2}/model_comparison_v2.json")
except subprocess.CalledProcessError:
    print("The comparison failed (see above). The results are still zipped, so it can be checked.")

collected = sorted(path for path in PHASE2.rglob("*") if path.is_file() and path.name != "artifact_manifest.json")
manifest = {"commit": GIT_COMMIT,
            "files": [{"path": path.relative_to(PHASE2).as_posix(), "bytes": path.stat().st_size,
                       "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in collected]}
(PHASE2 / "artifact_manifest.json").write_text(json.dumps(manifest, indent=1) + "\n", encoding="utf-8")
with zipfile.ZipFile("phase2_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PHASE2.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(PHASE2.parent))
print(f"{len(collected) + 1} files")
files.download("phase2_results.zip")